# 03B · Preparación de datos de siniestros con víctimas
### Escenario metodológico comparable antes y después de la Ley 2251 de 2022

Este notebook reconstruye el dataset analítico utilizando exclusivamente siniestros clasificados como `Con Heridos` o `Con Muertos`. Se conserva por separado el escenario original para garantizar trazabilidad.

## Justificación

El artículo 16 de la Ley 2251 de 2022 modificó el artículo 143 de la Ley 769 de 2002. Desde julio de 2022, los accidentes con solamente daños materiales dejaron de requerir la elaboración del IPAT por parte de las autoridades. La Circular Externa 20224000000057 del Ministerio de Transporte, del 29 de septiembre de 2022, impartió instrucciones para su aplicación.

En la base, los registros `Solo Daños` disminuyen de 12.557 en 2022 a 1.115 en 2023, mientras los siniestros con heridos o muertos permanecen relativamente estables. Para evitar mezclar dos regímenes de observación, este escenario restringe el universo a eventos con víctimas.

Fuentes oficiales:

- Ley 2251 de 2022: https://www.secretariasenado.gov.co/senado/basedoc/ley_2251_2022.html
- Secretaría Distrital de Movilidad: https://www.movilidadbogota.gov.co/preguntas-frecuentes/donde-se-consultan-los-reportes-oficiales-de-accidentes-de-transito-en-los-que

## 1. Importación y rutas

In [1]:
from pathlib import Path
import itertools

import holidays
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
RAW_FILE = Path('../data/raw/base-anuario-de-siniestralidad-2024.xlsx')
PROCESSED_PATH = Path('../data/processed')
REPORTS_PATH = Path('../reports/data_quality')
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
REPORTS_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = PROCESSED_PATH / 'dataset_victimas_localidad_franja_fecha.parquet'

## 2. Carga y diagnóstico del cambio de cobertura

In [2]:
usecols = [
    'Codigo_Accidente', 'Fecha_Acc', 'AA_Acc', 'MM_Acc', 'Hora_Acc',
    'Localidad', 'Gravedad_Indicador_Tradicional'
]
siniestros = pd.read_excel(RAW_FILE, sheet_name='Siniestros', usecols=usecols)
siniestros['Fecha_Acc'] = pd.to_datetime(siniestros['Fecha_Acc'])
assert siniestros['Codigo_Accidente'].is_unique
print(f'Base original: {len(siniestros):,} registros')
print('Categorías de gravedad:')
print(siniestros['Gravedad_Indicador_Tradicional'].value_counts(dropna=False))

Base original: 278,614 registros
Categorías de gravedad:
Gravedad_Indicador_Tradicional
Solo Daños     160078
Con Heridos    113419
Con Muertos      5117
Name: count, dtype: int64


In [3]:
coverage_audit = (
    siniestros[siniestros['AA_Acc'].between(2018, 2024)]
    .groupby(['AA_Acc', 'Gravedad_Indicador_Tradicional'])
    .size().unstack(fill_value=0)
    .rename_axis(index='Anio')
)
coverage_audit['Total'] = coverage_audit.sum(axis=1)
coverage_audit['Con_Victimas'] = coverage_audit.get('Con Heridos', 0) + coverage_audit.get('Con Muertos', 0)
coverage_audit['Variacion_Solo_Danos'] = coverage_audit['Solo Daños'].pct_change()
coverage_audit['Variacion_Con_Victimas'] = coverage_audit['Con_Victimas'].pct_change()
coverage_audit.to_csv(REPORTS_PATH / 'comparacion_cobertura_por_gravedad.csv', encoding='utf-8-sig')
coverage_audit

Gravedad_Indicador_Tradicional,Con Heridos,Con Muertos,Solo Daños,Total,Con_Victimas,Variacion_Solo_Danos,Variacion_Con_Victimas
Anio,,,,,,,
2018,12614,501,23891,37006,13115,NaN,NaN
2019,12376,493,22130,34999,12869,-0.073710,-0.018757
2020,8570,370,13783,22723,8940,-0.377180,-0.305307
2021,11006,462,17387,28855,11468,0.261482,0.282774
2022,12355,541,12557,25453,12896,-0.277794,0.124520
2023,12454,545,1115,14114,12999,-0.911205,0.007987
2024,12343,569,1037,13949,12912,-0.069955,-0.006693


La estabilidad de `Con Heridos` y `Con Muertos` frente a la caída de `Solo Daños` respalda la construcción de un universo comparable. Esto no implica que todos los siniestros con víctimas sean observados, sino que su mecanismo de registro no fue afectado de la misma manera por la exclusión legal del IPAT para eventos de solo daños.

## 3. Selección del universo con víctimas

In [4]:
victim_categories = ['Con Heridos', 'Con Muertos']
sin = siniestros[
    siniestros['AA_Acc'].between(2018, 2024)
    & siniestros['Gravedad_Indicador_Tradicional'].isin(victim_categories)
].copy()
assert sin[['Localidad', 'Hora_Acc', 'Fecha_Acc']].isna().sum().sum() == 0
print(f'Siniestros con víctimas 2018–2024: {len(sin):,}')
display(sin.groupby(['AA_Acc', 'Gravedad_Indicador_Tradicional']).size().unstack(fill_value=0))

Siniestros con víctimas 2018–2024: 85,199


Gravedad_Indicador_Tradicional,Con Heridos,Con Muertos
AA_Acc,,
2018,12614,501
2019,12376,493
2020,8570,370
2021,11006,462
2022,12355,541
2023,12454,545
2024,12343,569


## 4. Franja horaria y conteo diario

In [5]:
def assign_time_slot(hour):
    if 0 <= hour < 6: return 'Madrugada'
    if 6 <= hour < 12: return 'Mañana'
    if 12 <= hour < 18: return 'Tarde'
    return 'Noche'

sin['Franja_Horaria'] = sin['Hora_Acc'].map(assign_time_slot)
sin['Fecha_Acc'] = sin['Fecha_Acc'].dt.normalize()
event_count = (sin.groupby(['Localidad', 'Franja_Horaria', 'Fecha_Acc'])
               .size().rename('Num_Accidentes').reset_index())
print(f'Combinaciones con al menos un siniestro con víctimas: {len(event_count):,}')

Combinaciones con al menos un siniestro con víctimas: 63,005


## 5. Grid completo y variables de calendario

In [6]:
localities = sorted(sin['Localidad'].unique())
time_slots = ['Madrugada', 'Mañana', 'Tarde', 'Noche']
dates = pd.date_range('2018-01-01', '2024-12-31', freq='D')
grid = pd.DataFrame(
    itertools.product(localities, time_slots, dates),
    columns=['Localidad', 'Franja_Horaria', 'Fecha_Acc']
)
dataset = grid.merge(event_count, on=['Localidad', 'Franja_Horaria', 'Fecha_Acc'], how='left')
dataset['Num_Accidentes'] = dataset['Num_Accidentes'].fillna(0).astype(int)
co_holidays = holidays.Colombia(years=range(2018, 2025))
dataset['Dia_Semana'] = dataset['Fecha_Acc'].dt.day_name()
dataset['Mes'] = dataset['Fecha_Acc'].dt.month
dataset['Es_Fin_de_Semana'] = dataset['Fecha_Acc'].dt.dayofweek.isin([5, 6]).astype(int)
dataset['Es_Festivo'] = dataset['Fecha_Acc'].dt.date.astype('object').isin(co_holidays).astype(int)
print(f'Grid: {len(dataset):,} filas; localidades={len(localities)}; fechas={len(dates)}')

Grid: 204,560 filas; localidades=20; fechas=2557


## 6. Variables históricas sin fuga de información

In [7]:
dataset = dataset.sort_values(['Localidad', 'Franja_Horaria', 'Fecha_Acc']).reset_index(drop=True)
grouped = dataset.groupby(['Localidad', 'Franja_Horaria'])['Num_Accidentes']
dataset['Accidentes_Prom_7d'] = grouped.transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
dataset['Accidentes_Prom_30d'] = grouped.transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())
dataset['Accidentes_Semana_Anterior'] = grouped.transform(lambda x: x.shift(7))
history_cols = ['Accidentes_Prom_7d', 'Accidentes_Prom_30d', 'Accidentes_Semana_Anterior']
for col in history_cols:
    dataset[f'Sin_Historial_{col}'] = dataset[col].isna().astype(int)
dataset[history_cols] = dataset[history_cols].fillna(0)

## 7. Etiqueta Alto Riesgo

El umbral es el percentil 66 del número diario de siniestros con víctimas, calculado por localidad y franja exclusivamente con 2018–2022. El mismo umbral congelado se aplica a 2023–2024. En este escenario, `Alto_Riesgo` significa frecuencia elevada de siniestros con personas heridas o fallecidas respecto al historial de esa localidad y franja.

In [8]:
TRAIN_END = pd.Timestamp('2022-12-31')
dataset['Periodo'] = np.where(dataset['Fecha_Acc'] <= TRAIN_END, 'train', 'test')
risk_thresholds = (
    dataset[dataset['Periodo'].eq('train')]
    .groupby(['Localidad', 'Franja_Horaria'])['Num_Accidentes']
    .quantile(2 / 3).rename('Umbral_P66_Train').reset_index()
)
dataset = dataset.merge(risk_thresholds, on=['Localidad', 'Franja_Horaria'], how='left', validate='many_to_one')
dataset['Alto_Riesgo'] = (dataset['Num_Accidentes'] > dataset['Umbral_P66_Train']).astype(int)
print('Distribución de umbrales:')
display(risk_thresholds['Umbral_P66_Train'].value_counts().sort_index())
display(dataset.groupby('Periodo')['Alto_Riesgo'].agg(['count', 'sum', 'mean']))

Distribución de umbrales:


Umbral_P66_Train
0.0    49
1.0    28
2.0     3
Name: count, dtype: int64

,count,sum,mean
Periodo,,,
test,58480,11118,0.190116
train,146080,24907,0.170502


### Control de sensibilidad de la etiqueta

Como los conteos son discretos, se informa cuántas combinaciones reciben umbral 0, 1 o 2. Un umbral 0 hace que `Alto_Riesgo=1` equivalga a la ocurrencia de al menos un siniestro con víctimas; un umbral 1 exige dos o más. Esta heterogeneidad forma parte de la definición relativa por localidad y franja y debe explicarse al interpretar el modelo.

## 8. Validación y exportación

In [9]:
dataset = dataset.drop(columns='Umbral_P66_Train')
final_cols = [
    'Localidad', 'Franja_Horaria', 'Fecha_Acc', 'Num_Accidentes', 'Dia_Semana', 'Mes',
    'Es_Fin_de_Semana', 'Es_Festivo', 'Accidentes_Prom_7d', 'Accidentes_Prom_30d',
    'Accidentes_Semana_Anterior', 'Sin_Historial_Accidentes_Prom_7d',
    'Sin_Historial_Accidentes_Prom_30d', 'Sin_Historial_Accidentes_Semana_Anterior',
    'Periodo', 'Alto_Riesgo'
]
dataset = dataset[final_cols]
key = ['Localidad', 'Franja_Horaria', 'Fecha_Acc']
assert dataset.shape == (204560, 16)
assert dataset.duplicated(key).sum() == 0
assert dataset.isna().sum().sum() == 0
assert dataset['Localidad'].nunique() == 20
assert dataset['Franja_Horaria'].nunique() == 4
assert dataset['Fecha_Acc'].nunique() == 2557
dataset['Fecha_Acc'] = dataset['Fecha_Acc'].astype(str)
dataset.to_parquet(OUTPUT_FILE, index=False)
print('Guardado:', OUTPUT_FILE.resolve())
print('Dimensiones:', dataset.shape)
print('Nulos:', int(dataset.isna().sum().sum()), '| Duplicados de clave:', int(dataset.duplicated(key).sum()))

Guardado: /Users/camilo/Library/CloudStorage/OneDrive-FundaciónUniversitariaKonradLorenz/Siniestralidad_Vial/reports/reproducibilidad/ejecucion_20260913/data/processed/dataset_victimas_localidad_franja_fecha.parquet
Dimensiones: (204560, 16)
Nulos: 0 | Duplicados de clave: 0


## Resultado

El archivo exportado conserva la misma interfaz de 16 columnas del dataset original, pero `Num_Accidentes`, las variables históricas y `Alto_Riesgo` se refieren exclusivamente a siniestros con heridos o muertos. Esto permite adaptar los notebooks de modelado y evaluación cambiando la ruta de entrada, sin mezclar los modelos de ambos escenarios.